# OmniVoice Project Studio - Google Colab (AI-native)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/master/notebooks/OmniVoice_Project_Studio_Colab.ipynb)

Production notebook: Gradio Web UI + REST API + SSE jobs + MCP from one OmniVoice Studio server.

Startup uses an exact-revision persistent cache, active generation stays on Colab local SSD, and the notebook includes a cold/warm acceptance checkpoint.


In [ ]:
# Resolve immutable source first. Cached revisions are never selected implicitly.
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import re
import sys
import urllib.request

PACKAGE_REF = os.environ.get("OMNIVOICE_PACKAGE_REF", "").strip().lower()
if PACKAGE_REF:
    if re.fullmatch(r"[0-9a-f]{40}", PACKAGE_REF) is None:
        raise RuntimeError("OMNIVOICE_PACKAGE_REF must be an exact 40-character commit SHA.")
else:
    try:
        with urllib.request.urlopen(
            "https://api.github.com/repos/binhminhanh1235/OmniVoice/branches/master",
            timeout=20,
        ) as response:
            PACKAGE_REF = str(json.load(response)["commit"]["sha"]).lower()
    except Exception as exc:
        raise RuntimeError(
            "Cannot resolve current OmniVoice master exactly. Enable Internet for the small "
            "revision lookup or set OMNIVOICE_PACKAGE_REF to a verified 40-character commit SHA. "
            "A cached last_package_ref is intentionally never reused automatically."
        ) from exc
    if re.fullmatch(r"[0-9a-f]{40}", PACKAGE_REF) is None:
        raise RuntimeError("GitHub returned a non-immutable package revision.")

BOOTSTRAP_URL = (
    "https://raw.githubusercontent.com/binhminhanh1235/OmniVoice/"
    f"{PACKAGE_REF}/notebooks/hosted_runtime_bootstrap.py"
)
try:
    with urllib.request.urlopen(BOOTSTRAP_URL, timeout=30) as response:
        bootstrap_source = response.read().decode("utf-8")
except Exception as exc:
    raise RuntimeError(
        f"Cannot load the hosted-runtime bootstrap from exact revision {PACKAGE_REF}."
    ) from exc
exec(compile(bootstrap_source, BOOTSTRAP_URL, "exec"), globals(), globals())
globals().update(bootstrap_hosted_runtime(PACKAGE_REF))
del bootstrap_source

import torch
from omnivoice.hardware_quality import detect_hardware

print("CUDA:", torch.cuda.is_available())
hardware = detect_hardware()
print(hardware.summary())
for note in hardware.notes:
    print("-", note)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime -> Change runtime type -> T4 GPU")


## Local-first workspace + persistent Drive mirror

Generation runs from `/content/OmniVoiceStudio` on Colab local SSD. Google Drive stores the persistent project copy and startup cache, but it is not the render hot path.

Startup evidence is archived under `MyDrive/OmniVoiceStudio/.startup-evidence/`. Runtime cache metadata/evidence are excluded from the normal project mirror so a previous session cannot overwrite the current measurement.


In [ ]:
import atexit
import threading
from datetime import datetime, timezone

SYNC_INTERVAL_SECONDS = 45
Path(PERSISTENT_WORKSPACE).mkdir(parents=True, exist_ok=True)
evidence_archive = Path(PERSISTENT_WORKSPACE) / ".startup-evidence"
evidence_archive.mkdir(parents=True, exist_ok=True)
evidence_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
evidence_copy = evidence_archive / f"{evidence_stamp}-{PACKAGE_REF[:12]}.json"
shutil.copy2(STARTUP_CACHE_EVIDENCE, evidence_copy)
print("Archived startup evidence:", evidence_copy)

def _sync_workspace_tree(source, destination, *, delete=False):
    source_path = Path(source)
    destination_path = Path(destination)
    source_path.mkdir(parents=True, exist_ok=True)
    destination_path.mkdir(parents=True, exist_ok=True)
    rsync = shutil.which("rsync")
    if rsync:
        command = [
            rsync, "-a",
            "--exclude=.startup-cache/",
            "--exclude=.startup-evidence/",
            "--exclude=.runtime-cache.json",
            "--exclude=startup-cache-evidence.json",
            "--exclude=*.tmp",
            "--exclude=.nfs*",
        ]
        if delete:
            command.append("--delete")
        command.extend([f"{source_path}/", f"{destination_path}/"])
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL)
        return
    shutil.copytree(
        source_path,
        destination_path,
        dirs_exist_ok=True,
        symlinks=False,
        ignore=shutil.ignore_patterns(
            ".startup-cache", ".startup-evidence", ".runtime-cache.json",
            "startup-cache-evidence.json", "*.tmp", ".nfs*"
        ),
    )

if not globals().get("_OMNIVOICE_LOCAL_RESTORED", False):
    _sync_workspace_tree(PERSISTENT_WORKSPACE, WORKSPACE, delete=False)
    _OMNIVOICE_LOCAL_RESTORED = True
    print("Restored persistent Studio data to local SSD.")

old_stop = globals().get("_OMNIVOICE_SYNC_STOP")
if old_stop is not None:
    old_stop.set()
_OMNIVOICE_SYNC_STOP = threading.Event()

def sync_workspace_to_drive():
    _sync_workspace_tree(WORKSPACE, PERSISTENT_WORKSPACE, delete=True)

def _mirror_loop():
    while not _OMNIVOICE_SYNC_STOP.wait(SYNC_INTERVAL_SECONDS):
        try:
            sync_workspace_to_drive()
        except Exception as exc:
            print("Workspace mirror warning:", type(exc).__name__, exc)

_OMNIVOICE_SYNC_THREAD = threading.Thread(
    target=_mirror_loop, name="omnivoice-drive-mirror", daemon=True
)
_OMNIVOICE_SYNC_THREAD.start()
atexit.register(sync_workspace_to_drive)
write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)
print("Execution workspace:", WORKSPACE)
print("Persistent mirror:", PERSISTENT_WORKSPACE)
print(f"Mirror interval: {SYNC_INTERVAL_SECONDS}s")


## Production acceptance checkpoint

Set `ACCEPTANCE_SAMPLE` to `cold` only on a genuine cold run. Restart the runtime with the persistent cache intact, then use `warm`. Leave it empty when you are not recording an acceptance sample.

Evidence is stored under an exact package-revision directory in Drive. When both samples exist, this cell downloads the acceptance script from the same exact revision and executes it.


In [ ]:
from omnivoice.lazy_asr import should_defer_asr_startup

evidence = json.loads(Path(STARTUP_CACHE_EVIDENCE).read_text(encoding="utf-8"))
if evidence.get("package_ref") != PACKAGE_REF:
    raise RuntimeError("Startup evidence package_ref does not match the exact notebook package revision.")

ASR_DEVICE = "cpu"
print("Package ref:", PACKAGE_REF)
print("Bootstrap seconds:", evidence.get("bootstrap_seconds"))
print("Resource fast path:", evidence.get("resource_fast_path"))
print("Wheel fast path:", evidence.get("wheel_fast_path"))

lazy_cpu_expected = should_defer_asr_startup(ASR_DEVICE)
if not lazy_cpu_expected:
    raise RuntimeError("Colab production notebook expects CPU ASR to use lazy startup policy.")
print("ASR startup policy: lazy CPU")
print("Expected server startup log: CPU ASR startup deferred until first transcription/verification request.")
print("Expected first-use log: Initializing ASR on first use: ... device=cpu")

ACCEPTANCE_SAMPLE = ""  # set to "cold" or "warm" only for a genuine sample
sample = ACCEPTANCE_SAMPLE.strip().lower()
if sample not in {"", "cold", "warm"}:
    raise ValueError("ACCEPTANCE_SAMPLE must be empty, 'cold', or 'warm'.")

acceptance_root = evidence_archive / "acceptance" / PACKAGE_REF
acceptance_root.mkdir(parents=True, exist_ok=True)
if sample:
    sample_path = acceptance_root / f"{sample}.json"
    shutil.copy2(STARTUP_CACHE_EVIDENCE, sample_path)
    print("Recorded acceptance sample:", sample_path)

cold_path = acceptance_root / "cold.json"
warm_path = acceptance_root / "warm.json"
if cold_path.is_file() and warm_path.is_file():
    acceptance_url = (
        "https://raw.githubusercontent.com/binhminhanh1235/OmniVoice/"
        f"{PACKAGE_REF}/scripts/hosted_cache_acceptance.py"
    )
    acceptance_script = Path("/content/hosted_cache_acceptance.py")
    with urllib.request.urlopen(acceptance_url, timeout=30) as response:
        acceptance_script.write_bytes(response.read())
    subprocess.run(
        [sys.executable, str(acceptance_script), str(cold_path), str(warm_path)],
        check=True,
    )
else:
    print("Cold/warm pair not complete yet. Record only genuine samples.")

print("Acceptance evidence directory:", acceptance_root)


## Optional stable hostname + private access

Set `USE_STABLE_TUNNEL = True` only after creating a remotely managed Cloudflare Tunnel pointing your hostname to `http://localhost:8000`. Create Colab Secrets `CLOUDFLARE_TUNNEL_TOKEN`, `OMNIVOICE_API_TOKEN`, `OMNIVOICE_UI_USERNAME`, and `OMNIVOICE_UI_PASSWORD`.


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = False

if USE_STABLE_TUNNEL:
    from google.colab import userdata
    required_names = [
        "CLOUDFLARE_TUNNEL_TOKEN", "OMNIVOICE_API_TOKEN",
        "OMNIVOICE_UI_USERNAME", "OMNIVOICE_UI_PASSWORD",
    ]
    required = {}
    for name in required_names:
        try:
            required[name] = userdata.get(name)
        except Exception:
            required[name] = None
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /content/cloudflared
    !/content/cloudflared --version


## Launch

Stable mode exposes Gradio `/ui`, REST, SSE, and MCP through one server. Otherwise the notebook uses a temporary Gradio share URL. With CPU ASR, startup should log that ASR initialization is deferred until first transcription/verification.


In [ ]:
try:
    if USE_STABLE_TUNNEL:
        !omnivoice-studio serve \
          --model k2-fsa/OmniVoice \
          --workspace "$WORKSPACE" \
          --asr-model openai/whisper-small.en \
          --asr-device cpu \
          --host 0.0.0.0 \
          --port 8000 \
          --tunnel \
          --cloudflared /content/cloudflared \
          --public-url "$OMNIVOICE_PUBLIC_URL"
    else:
        !omnivoice-project-studio \
          --model k2-fsa/OmniVoice \
          --workspace "$WORKSPACE" \
          --asr-model openai/whisper-small.en \
          --asr-device cpu \
          --share
finally:
    _OMNIVOICE_SYNC_STOP.set()
    try:
        sync_workspace_to_drive()
        print("Final Studio workspace sync completed.")
    finally:
        persist_runtime_cache(CACHE_PREPARATION)
        print("Startup/model cache persisted.")
